# Framing Drift Over Time

Analyzing how frame usage changes over time, globally and by topic.

In [ ]:
!pip install pandas numpy matplotlib seaborn pyarrow scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_parquet('data/merged_topic_and_frames_filtered.parquet')
df['date'] = pd.to_datetime(df['date'])
df['quarter'] = df['date'].dt.to_period('Q')
print(f'Loaded {len(df):,} articles')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')

frames = ['economic', 'fairness', 'public_op', 'political', 'quality_life', 
          'crime', 'culture', 'health', 'legality', 'morality', 
          'policy', 'regulation', 'security', 'cap&res']

frame_matrix = np.array(df['vector'].tolist())
for i, frame in enumerate(frames):
    df[f'frame_{frame}'] = frame_matrix[:, i]

frame_cols = [f'frame_{f}' for f in frames]

## Global Frame Drift Over Time

In [ ]:
quarterly_global = df.groupby('quarter')[frame_cols].mean()

fig, ax = plt.subplots(figsize=(12, 6))
quarterly_global.plot(ax=ax)
plt.title('Global Frame Usage Over Time')
plt.xlabel('Quarter')
plt.ylabel('Usage Rate')
plt.legend(labels=frames, bbox_to_anchor=(1.02, 1), title='Frame')
plt.tight_layout()
plt.show()

In [ ]:
# linear regression to measure trend
x = np.arange(len(quarterly_global))

trends = []
for frame_col in frame_cols:
    y = quarterly_global[frame_col].values
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    trends.append({
        'frame': frame_col.replace('frame_', ''),
        'slope_per_quarter': slope,
        'slope_per_year': slope * 4,
        'r_squared': r_value ** 2,
        'p_value': p_value
    })

trend_df = pd.DataFrame(trends).sort_values('slope_per_year', key=abs, ascending=False)
trend_df.style.format({
    'slope_per_quarter': '{:+.2%}',
    'slope_per_year': '{:+.2%}',
    'r_squared': '{:.3f}',
    'p_value': '{:.4f}'
})

## Frame Drift by Topic

In [ ]:
topics = df['topic_top1'].value_counts().index.tolist()
print(f'Topics: {topics}')

In [ ]:
def plot_topic_drift(topic):
    subset = df[df['topic_top1'] == topic]
    quarterly = subset.groupby('quarter')[frame_cols].mean()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    quarterly.plot(ax=ax)
    plt.title(f'Frame Usage Over Time: {topic} (n={len(subset):,})')
    plt.xlabel('Quarter')
    plt.ylabel('Usage Rate')
    plt.legend(labels=frames, bbox_to_anchor=(1.02, 1), title='Frame')
    plt.tight_layout()
    plt.show()

for topic in topics[:4]:
    plot_topic_drift(topic)

In [ ]:
# trend by topic
topic_trends = []
for topic in topics:
    subset = df[df['topic_top1'] == topic]
    quarterly = subset.groupby('quarter')[frame_cols].mean()
    x = np.arange(len(quarterly))
    
    for frame_col in frame_cols:
        y = quarterly[frame_col].values
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        topic_trends.append({
            'topic': topic,
            'frame': frame_col.replace('frame_', ''),
            'slope_per_year': slope * 4,
            'p_value': p_value
        })

topic_trend_df = pd.DataFrame(topic_trends)
pivot = topic_trend_df.pivot(index='topic', columns='frame', values='slope_per_year')
pivot.style.format('{:+.1%}').background_gradient(cmap='PiYG', vmin=-0.05, vmax=0.05)

In [ ]:
# Summary of per-topic trend magnitudes
print('Summary of per-topic frame drift magnitudes:\n')

# Get absolute slopes for each topic-frame combination
abs_slopes = topic_trend_df['slope_per_year'].abs()
print(f'Overall range of per-topic trends: {abs_slopes.min():.1%} to {abs_slopes.max():.1%} per year')
print(f'Median absolute trend: {abs_slopes.median():.1%} per year')
print(f'Mean absolute trend: {abs_slopes.mean():.1%} per year')

# Show topics with largest trends
print('\nTopics with largest frame trends (any frame):')
max_by_topic = topic_trend_df.groupby('topic')['slope_per_year'].apply(lambda x: x.abs().max())
for topic in max_by_topic.sort_values(ascending=False).head(5).index:
    max_val = max_by_topic[topic]
    print(f'  {topic}: up to {max_val:.1%}/year')

# Count how many topic-frame combinations exceed thresholds
print(f'\nTopic-frame combinations with |trend| > 3%/year: {(abs_slopes > 0.03).sum()} of {len(abs_slopes)}')
print(f'Topic-frame combinations with |trend| > 5%/year: {(abs_slopes > 0.05).sum()} of {len(abs_slopes)}')

## Frame Drift by Outlet (for selected topic)

In [ ]:
selected_topic = 'elections and politics'
topic_df = df[df['topic_top1'] == selected_topic]
print(f'Analyzing: {selected_topic} ({len(topic_df):,} articles)')
print(f'\nArticles per outlet:')
print(topic_df['outlet_name'].value_counts())

In [ ]:
selected_frame = 'fairness'
frame_col = f'frame_{selected_frame}'
outlets = topic_df['outlet_name'].value_counts().head(6).index.tolist()

fig, ax = plt.subplots(figsize=(12, 6))
for outlet in outlets:
    outlet_df = topic_df[topic_df['outlet_name'] == outlet]
    quarterly = outlet_df.groupby('quarter')[frame_col].mean()
    ax.plot(quarterly.index.astype(str), quarterly.values, label=outlet, marker='o', markersize=4)

plt.title(f'{selected_frame.title()} Frame Usage Over Time by Outlet\nTopic: {selected_topic}')
plt.xlabel('Quarter')
plt.ylabel('Usage Rate')
plt.legend(title='Outlet')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
key_frames = ['fairness', 'political', 'legality', 'culture']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, frame in enumerate(key_frames):
    frame_col = f'frame_{frame}'
    ax = axes[idx]
    
    for outlet in outlets:
        outlet_df = topic_df[topic_df['outlet_name'] == outlet]
        quarterly = outlet_df.groupby('quarter')[frame_col].mean()
        ax.plot(quarterly.index.astype(str), quarterly.values, label=outlet, marker='o', markersize=3)
    
    ax.set_title(f'{frame.title()} Frame')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Usage Rate')
    ax.tick_params(axis='x', rotation=45)

axes[0].legend(title='Outlet', bbox_to_anchor=(1.02, 1))
plt.suptitle(f'Frame Drift by Outlet - Topic: {selected_topic}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
outlet_stats = []
for outlet in outlets:
    outlet_df = topic_df[topic_df['outlet_name'] == outlet]
    for frame in key_frames:
        frame_col = f'frame_{frame}'
        quarterly = outlet_df.groupby('quarter')[frame_col].mean()
        outlet_stats.append({
            'outlet': outlet,
            'frame': frame,
            'mean': quarterly.mean(),
            'std': quarterly.std(),
            'range': quarterly.max() - quarterly.min()
        })

outlet_stats_df = pd.DataFrame(outlet_stats)

for col in ['mean', 'std', 'range']:
    print(f'{col.upper()} frame usage over time:')
    pivot = outlet_stats_df.pivot(index='outlet', columns='frame', values=col)
    display(pivot.style.format('{:.1%}').background_gradient(cmap='Blues', axis=None))
    print()